# Exercise_0
- More EDAs on Olympic data

#### a) Start with reading in the dataset into a dataframe using spark.

In [0]:

DATA_PATH = "/Volumes/data/olympic_games/rawr_data"

df_athletes = spark.read.csv(F"{DATA_PATH}/athlete_events.csv", header=True, inferSchema=True)

####  b) Use spark columns method to find out the columns

In [0]:
df_athletes.columns

####  c) Find out the 10 oldest atheletes, their age and the sport

In [0]:
from pyspark.sql.types import StructField, StructType, StringType, IntegerType, FloatType, ShortType, ByteType

schema = StructType([
  StructField("ID", IntegerType(), True),
  StructField("Name", StringType(), True),
  StructField("Sex", StringType(), True),
  StructField("Age", ByteType(), True),
  StructField("Height", ShortType(), True),
  StructField("Weight", ShortType(), True),
  StructField("Team", StringType(), True),
  StructField("NOC", StringType(), True),
  StructField("Games", StringType(), True),
  StructField("Year", ShortType(), True),
  StructField("Season", StringType(), True),
  StructField("City", StringType(), True),
  StructField("Sport", StringType(), True),
  StructField("Event", StringType(), True),
  StructField("Medal", StringType(), True)
])

df_athletes_schema = spark.read.csv(f"{DATA_PATH}/athlete_events.csv", header=True, schema=schema, nullValue='NA')

df_athletes_schema.createOrReplaceTempView("df_athletes_schema")

spark.sql("""
          SELECT
          name,
          age,
          sport
          FROM df_athletes_schema
          GROUP BY name, age, sport
          ORDER BY age DESC
          LIMIT 10
          """).display()


#### d) Find out the 10 youngest atheletes, their age and the sport

In [0]:
spark.sql("""
          SELECT
          name,
          age,
          sport
          FROM df_athletes_schema
          GROUP BY name, age, sport
          ORDER BY age NULLS LAST
          LIMIT 10
          """).display()

#### e) Find out the five sports with highest median age

In [0]:
spark.sql("""       
SELECT
sport,
percentile_approx(age, 0.5) AS median_age
FROM df_athletes_schema
WHERE age IS NOT NULL
GROUP BY sport
ORDER BY median_age DESC
 """).display()


  f) Find out the five sports with lowest median age

In [0]:
spark.sql("""       
SELECT
sport,
percentile_approx(age, 0.5) AS median_age
FROM df_athletes_schema
WHERE age IS NOT NULL
GROUP BY sport
ORDER BY median_age
 """).display()

#### g) Find out top 10 countries after number of gold medals

In [0]:
spark.sql("""       
SELECT
noc,
count(medal) as total_medals
FROM df_athletes_schema
WHERE medal = 'Gold'
GROUP BY noc, medal
ORDER BY total_medals DESC
LIMIT 10;
""").display()

#### h) Find out top 10 countries after number of medals

In [0]:
spark.sql("""       
SELECT
noc,
count(medal) as total_medals
FROM df_athletes_schema
WHERE medal = 'bronze' OR medal = 'silver' OR medal = 'Gold'
GROUP BY noc, medal
ORDER BY total_medals DESC
LIMIT 10;
""").display()

#### i) Plot a time series line chart of number of female and male atheletes in same graph.

In [0]:
df_genders = spark.sql("""
SELECT
year,
sex, count(*) as total_count
FROM df_athletes_schema
WHERE Sex = 'M' OR Sex = 'F'
GROUP BY year, sex
Order BY year
                          """)
df_genders.show()

In [0]:
df_pivot = df_genders.toPandas().pivot(index="year", columns="sex", values="total_count")
df_pivot.plot(kind="line")

#### j) Do more explorations on your own

In [0]:
df_common_city = spark.sql("""
SELECT sport, city, count(*) as most_participants_by_sport
FROM df_athletes_schema
WHERE city IS NOT NULL
GROUP BY city, sport
Order by most_participants_by_sport DESC
LIMIT 10
                          """)
display(df_common_city)

## 1. Upload data files to Databricks

#### a) Create a new catalog, airbnb

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS airbnb;

#### b) Create a schema, hosts

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS hosts;

#### c) Create a volume, csv_files and upload the csv file from Kaggle here

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS csv_files;



In [0]:
df = spark.read.csv(
    "/Volumes/airbnb/hosts/csv_files/Airbnb_Open_Data.csv", header=True, inferSchema=True
)

#### d) Now create an ETL pipeline for SDP. This pipeline should be able to produce a streaming table under the same schema created in question b) above. Use pyspark for this.

In [0]:
df = spark.read.csv(
    "/Volumes/airbnb/hosts/csv_files/Airbnb_Open_Data.csv",
    header=True
)
print(df.columns)